# 📈 Lineer Modeller (Linear Models)

Özellikle House Prices gibi projelerde veya Stacking yaparken 'Meta Model' olarak sıklıkla kullanılan cezalandırmalı (Regularized) regresyon modelleri.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.model_selection import GridSearchCV

## 1. Ridge Regresyon (L2 Penalty)
Çoklu bağlantı (Multicollinearity) sorununu azaltır. Özelliklerin ağırlıklarını küçültür ama tamamen sıfırlamaz.

In [ ]:
def train_ridge_with_cv(X_train, y_train, X_test):
    """
    GridSearchCV ile en iyi alpha (ceza) değerini bularak Ridge modelini eğitir.
    Not: Lineer modellerde X_train mutlaka Ölçeklendirilmiş (Scaled) olmalıdır!
    """
    ridge = Ridge(random_state=42)
    
    # Aranacak alpha (ceza) parametreleri
    param_grid = {'alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
    
    grid_search = GridSearchCV(estimator=ridge, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    
    print("Ridge En İyi Alpha:", grid_search.best_params_)
    
    best_model = grid_search.best_estimator_
    preds = best_model.predict(X_test)
    
    return best_model, preds

## 2. Lasso Regresyon (L1 Penalty)
Aynı zamanda bir Özellik Seçici (Feature Selector) görevi görür. Önemsiz özelliklerin katsayılarını tamamen SIFIR yapar.

In [ ]:
def train_lasso_with_cv(X_train, y_train, X_test):
    """
    Lasso ile en iyi alpha değerini bularak modeli eğitir.
    """
    # max_iter'i yüksek tutmak yakınsama (convergence) sorunlarını önler
    lasso = Lasso(random_state=42, max_iter=10000)
    
    param_grid = {'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}
    
    grid_search = GridSearchCV(estimator=lasso, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    
    print("Lasso En İyi Alpha:", grid_search.best_params_)
    
    best_model = grid_search.best_estimator_
    preds = best_model.predict(X_test)
    
    # Sıfır olan katsayıları görelim (Modelin attığı özellikler)
    zero_coefs = sum(best_model.coef_ == 0)
    print(f"Lasso {zero_coefs} özelliği sıfırladı (iptal etti).")
    
    return best_model, preds

## 3. ElasticNet (L1 + L2)
Lasso ve Ridge'in karışımıdır. Hem özellikleri sıfırlayabilir hem de ağırlıklarını küçültebilir.

In [ ]:
def train_elasticnet_with_cv(X_train, y_train, X_test):
    """
    l1_ratio: L1 (Lasso) cezasının ağırlığı. 1 olursa tam Lasso, 0 olursa tam Ridge olur.
    """
    elastic = ElasticNet(random_state=42, max_iter=10000)
    
    param_grid = {
        'alpha': [0.01, 0.1, 1.0, 10.0],
        'l1_ratio': [0.2, 0.5, 0.8] # Hem ridge hem lasso etkisini arar
    }
    
    grid_search = GridSearchCV(estimator=elastic, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    
    print("ElasticNet En İyi Parametreler:", grid_search.best_params_)
    
    best_model = grid_search.best_estimator_
    preds = best_model.predict(X_test)
    
    return best_model, preds